# 📖 Notebook 1: Distributed Coordination

Imagine 3 servers all trying to process the same payment at the same time. Without coordination, the customer gets charged 3 times. **Distributed locks** prevent this — they ensure only one server processes the payment at a time.

In this notebook, we'll see three approaches:

| Approach | Method | Problem |
|----------|--------|---------|
| 🔴 Bad | No coordination | Race conditions, data corruption |
| 🟡 Better | File-based locks | Works on one machine only, no crash recovery |
| 🟢 Best | ZooKeeper distributed locks | Works across machines, auto-releases on crash |

## Learning Objectives

By the end of this notebook, you'll understand:
- Why race conditions happen in distributed systems
- Why simple locks don't work across multiple machines
- How ZooKeeper ephemeral nodes solve the problem
- What happens when a lock holder crashes

## 🛠️ Setup

Start the ZooKeeper ensemble first:

```bash
cd 03-technologies/coordination/zookeeper
docker compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import threading
import time
import os
import tempfile

# We'll use this shared variable to simulate a bank account balance
# In the real world, this would be a database row
shared_balance = 0
num_workers = 5
deposits_per_worker = 20

---
## 🔴 Bad: No Coordination (Race Conditions)

### The Problem

When multiple workers access the same resource without coordination, they step on each other's toes.

Imagine a bank account with $1,000. Two servers both try to withdraw $100 at the same time:

```
Server A reads balance: $1,000
Server B reads balance: $1,000      ← both see the same value!
Server A writes: $1,000 - $100 = $900
Server B writes: $1,000 - $100 = $900   ← Server B overwrites A's result!

Expected final balance: $800
Actual final balance:   $900  ← $100 disappeared into thin air!
```

This is called a **race condition** — the result depends on the timing of operations.

In [ ]:
def deposit_no_lock(worker_id, deposits, results):
    """
    BAD: Each worker reads and writes without any coordination.
    Multiple workers will overwrite each other's changes.
    """
    global shared_balance
    for i in range(deposits):
        # Step 1: Read the current balance (like reading from a database)
        current = shared_balance

        # Step 2: Simulate some processing time
        # In the real world, this could be network latency, validation, etc.
        time.sleep(0.001)

        # Step 3: Write the new balance
        # BUG: Another worker may have changed the balance between step 1 and 3!
        shared_balance = current + 1

    results[worker_id] = deposits


# Reset and run the experiment
shared_balance = 0
results = {}

# Launch 5 workers, each making 20 deposits of $1
threads = []
for i in range(num_workers):
    t = threading.Thread(target=deposit_no_lock, args=(i, deposits_per_worker, results))
    threads.append(t)
    t.start()

for t in threads:
    t.join()

expected = num_workers * deposits_per_worker
print(f"Expected balance: {expected}")
print(f"Actual balance:   {shared_balance}")
print(f"Lost updates:     {expected - shared_balance}")
print()
if shared_balance < expected:
    print("❌ RACE CONDITION! Some deposits were lost because workers overwrote each other.")
else:
    print("✅ Got lucky this time, but race conditions are non-deterministic. Run again!")

### 🤔 What Went Wrong?

Without locking, the classic **read-modify-write** pattern is broken:

```
Worker A: reads 10 → calculates 11 → writes 11
Worker B: reads 10 → calculates 11 → writes 11  ← overwrites A's write!
```

Both workers read the *same* value, did their own calculation, and wrote back. Worker B's write erased Worker A's deposit.

**In a distributed system, this is even worse** because the "workers" are on different machines with network latency between them.

---
## 🟡 Better: File-Based Locks

### The Idea

One classic fix is to use a **lock file**. Before accessing the shared resource, a worker creates a lock file. Other workers check for the file and wait if it exists.

This is like putting a "Do Not Disturb" sign on a hotel room door.

### Why It's Better
- Prevents race conditions on a single machine
- Simple to understand and implement

### Why It's Still Not Great
- **Only works on one machine** — can't share a file lock across servers
- **No crash recovery** — if a worker crashes while holding the lock, the lock file stays forever (deadlock!)
- **No fairness** — no guarantee that waiting workers get the lock in order

In [ ]:
class FileLock:
    """
    A simple file-based lock.
    Creates a file to indicate the lock is held.
    Deletes the file to release the lock.
    """

    def __init__(self, lock_path):
        self.lock_path = lock_path

    def acquire(self, timeout=10):
        """Try to create the lock file. Wait if it already exists."""
        start = time.time()
        while True:
            try:
                # os.open with O_CREAT | O_EXCL is atomic:
                # it creates the file ONLY if it doesn't exist
                fd = os.open(self.lock_path, os.O_CREAT | os.O_EXCL | os.O_WRONLY)
                os.close(fd)
                return True
            except FileExistsError:
                # Someone else holds the lock — wait and retry
                if time.time() - start > timeout:
                    return False
                time.sleep(0.01)

    def release(self):
        """Delete the lock file to release the lock."""
        try:
            os.remove(self.lock_path)
        except FileNotFoundError:
            pass  # already released


# Create a temporary lock file path
lock_file = os.path.join(tempfile.gettempdir(), "bank_account.lock")

# Clean up any stale lock from previous runs
if os.path.exists(lock_file):
    os.remove(lock_file)

print(f"Lock file location: {lock_file}")

In [ ]:
def deposit_with_file_lock(worker_id, deposits, results):
    """
    BETTER: Each worker acquires a file lock before accessing the shared resource.
    This prevents race conditions on a single machine.
    """
    global shared_balance
    lock = FileLock(lock_file)

    for i in range(deposits):
        # Step 1: Acquire the lock (wait if someone else has it)
        lock.acquire()
        try:
            # Step 2: Safely read and modify the balance
            current = shared_balance
            time.sleep(0.001)  # simulate processing
            shared_balance = current + 1
        finally:
            # Step 3: ALWAYS release the lock, even if an error occurs
            lock.release()

    results[worker_id] = deposits


# Reset and run the experiment
shared_balance = 0
results = {}

threads = []
for i in range(num_workers):
    t = threading.Thread(target=deposit_with_file_lock, args=(i, deposits_per_worker, results))
    threads.append(t)
    t.start()

for t in threads:
    t.join()

expected = num_workers * deposits_per_worker
print(f"Expected balance: {expected}")
print(f"Actual balance:   {shared_balance}")
print()
if shared_balance == expected:
    print("✅ File lock prevented race conditions! All deposits accounted for.")
else:
    print("❌ Something went wrong.")

### ⚠️ The Fatal Flaw: What If the Lock Holder Crashes?

Let's simulate a worker crashing while holding the lock:

In [ ]:
# Clean up any stale lock
if os.path.exists(lock_file):
    os.remove(lock_file)

lock = FileLock(lock_file)

# Simulate: Worker acquires lock then crashes (never releases it)
print("Worker A acquires the lock...")
lock.acquire()
print("Worker A 'crashes' (never calls release)!")
print()

# Now another worker tries to get the lock
print("Worker B tries to acquire the lock (timeout=3s)...")
lock2 = FileLock(lock_file)
got_lock = lock2.acquire(timeout=3)
print(f"Worker B got the lock: {got_lock}")
print()
if not got_lock:
    print("💀 DEADLOCK! Worker B can never proceed because the lock file is stuck.")
    print("   In production, this means your entire system stops processing.")
    print("   Someone has to manually delete the lock file to recover.")

# Clean up
if os.path.exists(lock_file):
    os.remove(lock_file)

### Why File Locks Fail in Distributed Systems

| Limitation | Why It Matters |
|------------|----------------|
| Single machine only | Can't coordinate across servers |
| No crash recovery | Lock stays forever if holder crashes |
| No fairness | Starvation possible — some workers may never get the lock |
| No monitoring | Hard to know who holds the lock or how long they've had it |

---
## 🟢 Best: ZooKeeper Distributed Locks

### The Solution

ZooKeeper solves all the problems above using **ephemeral nodes**:

1. To acquire a lock, a client creates an **ephemeral node** in ZooKeeper
2. "Ephemeral" means the node is **automatically deleted** when the client disconnects
3. If a lock holder crashes, its connection drops, the ephemeral node vanishes, and the lock is released!

```
ZooKeeper before crash:         ZooKeeper after crash:
/locks                          /locks
  └── lock-0001 (Worker A)        └── (empty — lock auto-released!)
```

### Why This Is Better Than File Locks

| Feature | File Lock | ZooKeeper Lock |
|---------|-----------|----------------|
| Works across machines | ❌ | ✅ |
| Crash recovery | ❌ Deadlock | ✅ Auto-release |
| Fairness (FIFO) | ❌ | ✅ |
| Monitoring | ❌ | ✅ |

In [ ]:
from kazoo.client import KazooClient

# Connect to our 3-node ZooKeeper ensemble
# If one node goes down, the client automatically reconnects to another
zk = KazooClient(hosts="localhost:2181,localhost:2182,localhost:2183")
zk.start()

print(f"Connected to ZooKeeper!")
print(f"Session ID: {hex(zk.client_id[0])}")

In [ ]:
def deposit_with_zk_lock(worker_id, deposits, results, zk_hosts):
    """
    BEST: Each worker acquires a ZooKeeper distributed lock.
    - Works across multiple machines
    - Lock auto-releases if the worker crashes
    - Fair ordering (FIFO) — workers get the lock in request order
    """
    global shared_balance

    # Each worker gets its own ZooKeeper connection
    worker_zk = KazooClient(hosts=zk_hosts)
    worker_zk.start()

    # The Lock recipe creates ephemeral sequential nodes under this path
    lock = worker_zk.Lock("/demo/locks/bank-account", f"worker-{worker_id}")

    for i in range(deposits):
        # acquire() blocks until this worker has the lock
        with lock:
            current = shared_balance
            time.sleep(0.001)  # simulate processing
            shared_balance = current + 1

    results[worker_id] = deposits
    worker_zk.stop()


# Reset and run the experiment
shared_balance = 0
results = {}
zk_hosts = "localhost:2181,localhost:2182,localhost:2183"

threads = []
for i in range(num_workers):
    t = threading.Thread(
        target=deposit_with_zk_lock,
        args=(i, deposits_per_worker, results, zk_hosts)
    )
    threads.append(t)
    t.start()

for t in threads:
    t.join()

expected = num_workers * deposits_per_worker
print(f"Expected balance: {expected}")
print(f"Actual balance:   {shared_balance}")
print()
if shared_balance == expected:
    print("✅ ZooKeeper lock prevented race conditions! All deposits accounted for.")
else:
    print("❌ Something went wrong.")

### 🔍 Let's Peek Inside ZooKeeper

Let's see what the lock nodes actually look like:

In [ ]:
# Create a lock and hold it so we can inspect the ZooKeeper nodes
inspect_lock = zk.Lock("/demo/locks/inspect", "inspector")
inspect_lock.acquire()

print("Lock nodes in ZooKeeper:")
print("========================")

children = zk.get_children("/demo/locks/inspect")
for child in sorted(children):
    data, stat = zk.get(f"/demo/locks/inspect/{child}")
    print(f"  /demo/locks/inspect/{child}")
    print(f"    Data: {data.decode()}")
    print(f"    Ephemeral: {stat.ephemeralOwner != 0}")
    print()

inspect_lock.release()
print("After release — lock nodes are automatically cleaned up.")
if zk.exists("/demo/locks/inspect"):
    remaining = zk.get_children("/demo/locks/inspect")
    print(f"  Remaining nodes: {len(remaining)}")

### 💥 Crash Recovery Demo

The killer feature of ZooKeeper locks: **if the lock holder crashes, the lock is automatically released**.

Let's prove it:

In [ ]:
# Worker A connects and acquires the lock
worker_a = KazooClient(hosts="localhost:2181,localhost:2182,localhost:2183")
worker_a.start()
lock_a = worker_a.Lock("/demo/locks/crash-test", "worker-a")
lock_a.acquire()
print("Worker A acquired the lock.")

# Show lock exists
children = zk.get_children("/demo/locks/crash-test")
print(f"Lock nodes before crash: {sorted(children)}")

# Worker B tries to get the lock in a background thread
worker_b_result = {"got_lock": False, "wait_time": 0}

def worker_b_task():
    worker_b = KazooClient(hosts="localhost:2181,localhost:2182,localhost:2183")
    worker_b.start()
    lock_b = worker_b.Lock("/demo/locks/crash-test", "worker-b")

    print("Worker B waiting for the lock...")
    start = time.time()
    lock_b.acquire()  # blocks until lock is available
    worker_b_result["wait_time"] = round(time.time() - start, 2)
    worker_b_result["got_lock"] = True
    print(f"Worker B got the lock after {worker_b_result['wait_time']}s!")
    lock_b.release()
    worker_b.stop()

b_thread = threading.Thread(target=worker_b_task)
b_thread.start()
time.sleep(1)

# Simulate Worker A crashing by closing its connection abruptly
print("\n💥 Worker A crashes! (connection closed)")
worker_a.stop()  # this simulates a crash — connection drops

# Wait for Worker B to get the lock
b_thread.join(timeout=15)
print()

if worker_b_result["got_lock"]:
    print("✅ ZooKeeper automatically released the lock when Worker A crashed!")
    print("   No manual intervention needed. No deadlock. The system self-healed.")
else:
    print("⏳ Worker B is still waiting (ZooKeeper session timeout can take up to 10s).")

---
## 📊 Summary: Bad → Better → Best

| | 🔴 No Coordination | 🟡 File Locks | 🟢 ZooKeeper Locks |
|---|---|---|---|
| **Correctness** | ❌ Race conditions | ✅ Prevents races | ✅ Prevents races |
| **Multi-machine** | N/A | ❌ Single machine | ✅ Any number of machines |
| **Crash recovery** | N/A | ❌ Deadlock | ✅ Auto-release |
| **Fairness** | N/A | ❌ Random | ✅ FIFO ordering |
| **Complexity** | None | Low | Medium (needs ZooKeeper cluster) |

### Key Takeaway

ZooKeeper's **ephemeral nodes** are the magic ingredient. When a client disconnects (intentionally or by crashing), its ephemeral nodes are automatically deleted. This is what makes distributed locks reliable — you never get stuck in a deadlock because of a crashed process.

### When to Use This

Use ZooKeeper distributed locks when:
- Multiple servers need exclusive access to a shared resource
- You need automatic recovery from crashes
- You need fair (FIFO) lock ordering

Examples: payment processing, inventory updates, file processing pipelines

In [ ]:
# Cleanup
if zk.exists("/demo"):
    zk.delete("/demo", recursive=True)
zk.stop()
print("Cleaned up ZooKeeper nodes. Done!")